# Load the sweep's results into DataFrames

Loading only. Nothing here interprets a number - the point is to get every
file in `results/` into a DataFrame with a `model` column, so analysis can
start from a clean slate in your own cells.

Runs locally in this repo's venv (`uv sync --extra dev`), not on Colab.
There is no GPU work and nothing is downloaded; it reads CSVs off disk.

Model folders are discovered by looking for `summary-row.json`, the same
marker the sweep's resume check uses, rather than from a typed list - so a
model added to the registry later shows up here without editing this
notebook.

## 1. Where the results are

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# The notebook lives in notebooks/local/, so the repo root is two up.
RESULTS = Path.cwd()
while not (RESULTS / "results").is_dir() and RESULTS != RESULTS.parent:
    RESULTS = RESULTS.parent
RESULTS = RESULTS / "results"

# A finished model is one the sweep wrote a summary row for.
MODEL_DIRS = sorted(p.parent for p in RESULTS.glob("*/summary-row.json"))

print(f"results  {RESULTS}")
print(f"models   {len(MODEL_DIRS)}")
for d in MODEL_DIRS:
    print(f"  {d.name}")

## 2. Cross-model tables

`summary` is the one row per model the sweep wrote; `metadata` is the
reproduction record beside it - model id, dtype, device, GPU, seed and the
full search config.

In [ ]:
summary_csv = RESULTS / "beam-search-teddy-bear.csv"
summary = pd.read_csv(summary_csv) if summary_csv.exists() else pd.DataFrame()

metadata = pd.DataFrame(
    json.loads((d / "run-metadata.json").read_text(encoding="utf-8"))
    for d in MODEL_DIRS
)

# The search config is one nested dict per row; spread it into columns so it
# can be compared across models without digging into the dict each time.
if "config" in metadata.columns:
    config = pd.json_normalize(metadata["config"]).add_prefix("config.")
    metadata = pd.concat([metadata.drop(columns=["config"]), config], axis=1)

print(f"summary   {summary.shape}")
print(f"metadata  {metadata.shape}")

## 3. Per-model tables, concatenated

Each of these is every model's copy of one file stacked into a single frame
with a `model` column in front, so you can `groupby("model")` or filter to
one and get the original table back.

| frame | file | one row per |
|---|---|---|
| `finals` | `final_results.csv` | returned sequence (4 per model) |
| `greedy` | `greedy_baseline.csv` | token of the width-1 run |
| `tokens` | `token-level-probabilities.csv` | token of a returned sequence |
| `traces` | `beam_trace.csv` | candidate at every step - the big one |

In [ ]:
def stack(filename):
    """Every model's copy of one CSV, with a leading `model` column."""
    frames = []
    for d in MODEL_DIRS:
        path = d / filename
        if not path.exists():
            print(f"  missing: {d.name}/{filename}")
            continue
        frame = pd.read_csv(path)
        frame.insert(0, "model", d.name)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


finals = stack("final_results.csv")
greedy = stack("greedy_baseline.csv")
tokens = stack("token-level-probabilities.csv")
traces = stack("beam_trace.csv")

for name, frame in (("finals", finals), ("greedy", greedy),
                    ("tokens", tokens), ("traces", traces)):
    print(f"{name:9} {str(frame.shape):>16}")

## 4. The weight-layout experiment

A separate, earlier experiment that reads safetensors headers rather than
running anything, written up in `docs/weight-layout-comparison.md`. It
covers a different set of models from the sweep - including two that were
later dropped - so it is loaded separately and not joined to the tables
above.

In [ ]:
arch_csv = RESULTS / "architecture-comparison.csv"
architecture = pd.read_csv(arch_csv) if arch_csv.exists() else pd.DataFrame()

manifest_paths = sorted(RESULTS.glob("*/tensor-manifest.csv"))
manifests = pd.concat(
    [
        pd.read_csv(p).assign(model=p.parent.name)
        for p in manifest_paths
    ],
    ignore_index=True,
) if manifest_paths else pd.DataFrame()

if not manifests.empty:
    manifests = manifests[["model", *[c for c in manifests.columns if c != "model"]]]

print(f"architecture  {architecture.shape}")
print(f"manifests     {manifests.shape}  "
      f"from {len(manifest_paths)} model(s)")

## 5. Anything the sweep skipped

Written only when a model failed. Present-but-stale is possible: it is not
cleared by a later run, so check it against `summary` before trusting it.

In [ ]:
skipped_csv = RESULTS / "skipped-models.csv"
skipped = pd.read_csv(skipped_csv) if skipped_csv.exists() else pd.DataFrame()

print(f"skipped  {skipped.shape}")
if not skipped.empty and not summary.empty:
    both = set(skipped["model"]) & set(summary["model"])
    for model in sorted(both):
        print(f"  stale: {model} is listed as skipped but has results")

## 6. What is loaded

In [ ]:
loaded = {
    "summary": summary,
    "metadata": metadata,
    "finals": finals,
    "greedy": greedy,
    "tokens": tokens,
    "traces": traces,
    "architecture": architecture,
    "manifests": manifests,
    "skipped": skipped,
}

pd.DataFrame(
    [
        {
            "frame": name,
            "rows": len(frame),
            "columns": len(frame.columns),
            "memory_MB": round(frame.memory_usage(deep=True).sum() / 1e6, 2),
        }
        for name, frame in loaded.items()
    ]
)

## 7. Columns, for reference

So you do not have to `.columns` each one while writing a query.

In [ ]:
for name, frame in loaded.items():
    if frame.empty:
        print(f"{name}: empty")
        continue
    print(f"{name}:")
    print(f"    {', '.join(frame.columns)}")
    print()

---

Everything above is loaded. Analysis goes below this line.

In [ ]:
# your analysis here